# 🧪 Test Backtesting System with Mini Dataset

**Dataset:** 3 ETFs (SPY, AGG, GLD), 1 Year (252 days)

**Purpose:** 
- ทดสอบว่าระบบทำงานได้จริง
- สร้าง sample charts
- ตรวจสอบ modules ทั้งหมด

---

## ⚠️ ก่อนรัน:
1. รัน `IMPORT_MINI_DATA.sql` ใน MySQL Workbench
2. รัน `python import_mini_price_history.py`

---

## Cell 1: Import Libraries

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import custom modules
sys.path.append('modules')
from data_loader import DataLoader
from risk_metrics import RiskMetrics
from portfolio_optimizer import PortfolioOptimizer
from performance_analytics import PerformanceAnalytics

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✅ Libraries loaded successfully")

---

## Cell 2: Load Data from MySQL

In [ ]:
# MySQL Configuration
MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',
    'database': 'portfolio_backtesting'
}

# Initialize DataLoader
loader = DataLoader(MYSQL_CONFIG)

print("📊 Loading data from MySQL...\n")

# Load ETFs
etfs = loader.get_etfs()
print(f"✅ Loaded {len(etfs)} ETFs")
print(etfs[['ticker_symbol', 'etf_name', 'asset_class']])

# Load Benchmarks
print("\n" + "="*70)
benchmarks = loader.get_benchmarks()
print(f"\n✅ Loaded {len(benchmarks)} Benchmarks")
print(benchmarks[['benchmark_name', 'description']])

# Load Price History
print("\n" + "="*70)
prices = loader.get_price_history()
print(f"\n✅ Loaded {len(prices):,} price history rows")
print(f"   Date range: {prices['price_date'].min()} to {prices['price_date'].max()}")
print(f"\n📋 Sample data:")
print(prices.head(10))

---

## Cell 3: Calculate Returns

In [ ]:
# Pivot to get close prices for each ticker
price_pivot = prices.pivot(index='price_date', columns='ticker_symbol', values='close_price')
price_pivot = price_pivot.sort_index()

print("📈 Price Data Shape:", price_pivot.shape)
print("\n📋 Last 5 days:")
print(price_pivot.tail())

# Calculate daily returns
returns = price_pivot.pct_change().dropna()

print("\n📊 Daily Returns Shape:", returns.shape)
print("\n📋 Statistics:")
print(returns.describe())

---

## Cell 4: Test Risk Metrics Module

In [ ]:
print("="*70)
print("📊 Testing Risk Metrics Module")
print("="*70)

risk_results = {}

for ticker in returns.columns:
    ticker_returns = returns[ticker].values
    
    sharpe = RiskMetrics.sharpe_ratio(ticker_returns)
    sortino = RiskMetrics.sortino_ratio(ticker_returns)
    max_dd = RiskMetrics.max_drawdown(ticker_returns)
    volatility = RiskMetrics.volatility(ticker_returns)
    
    risk_results[ticker] = {
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown': max_dd,
        'Volatility (Annual)': volatility
    }
    
    print(f"\n{ticker}:")
    print(f"  Sharpe Ratio:       {sharpe:7.2f}")
    print(f"  Sortino Ratio:      {sortino:7.2f}")
    print(f"  Max Drawdown:       {max_dd:7.2%}")
    print(f"  Volatility (Ann.):  {volatility:7.2%}")

# Create DataFrame
risk_df = pd.DataFrame(risk_results).T
print("\n" + "="*70)
print("\n📊 Risk Metrics Summary:")
print(risk_df)

---

## Cell 5: Test Portfolio Optimizer

In [ ]:
print("="*70)
print("📊 Testing Portfolio Optimizer Module")
print("="*70)

# Initialize optimizer
optimizer = PortfolioOptimizer(returns)

# Optimize for maximum Sharpe ratio
print("\n🎯 Optimizing for Maximum Sharpe Ratio...")
optimal_weights = optimizer.optimize_sharpe()

print("\n✅ Optimal Weights:")
for ticker, weight in optimal_weights.items():
    print(f"  {ticker}: {weight:6.2%}")

# Calculate portfolio metrics
portfolio_returns = (returns * pd.Series(optimal_weights)).sum(axis=1)
portfolio_sharpe = RiskMetrics.sharpe_ratio(portfolio_returns.values)
portfolio_volatility = RiskMetrics.volatility(portfolio_returns.values)

print(f"\n📊 Optimized Portfolio Metrics:")
print(f"  Sharpe Ratio:  {portfolio_sharpe:.2f}")
print(f"  Volatility:    {portfolio_volatility:.2%}")

---

## Cell 6: Visualize Price History

In [ ]:
# Normalize prices to 100
normalized_prices = price_pivot / price_pivot.iloc[0] * 100

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

for ticker in normalized_prices.columns:
    ax.plot(normalized_prices.index, normalized_prices[ticker], label=ticker, linewidth=2)

ax.set_title('ETF Performance (Normalized to 100)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Value (Base 100)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📈 Chart: ETF Performance Over Time")

---

## Cell 7: Correlation Heatmap

In [ ]:
# Calculate correlation matrix
corr_matrix = returns.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8})

ax.set_title('ETF Return Correlation Matrix', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Correlation Matrix:")
print(corr_matrix)

---

## Cell 8: Risk-Return Scatter Plot

In [ ]:
# Calculate annual returns and volatility
annual_returns = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#FF6B6B', '#4ECDC4', '#FFD93D']

for i, ticker in enumerate(returns.columns):
    ax.scatter(annual_volatility[ticker], annual_returns[ticker], 
               s=200, alpha=0.7, color=colors[i], label=ticker, edgecolors='black')
    ax.annotate(ticker, (annual_volatility[ticker], annual_returns[ticker]),
                xytext=(10, 10), textcoords='offset points', fontsize=12, fontweight='bold')

ax.set_title('Risk-Return Profile', fontsize=16, fontweight='bold')
ax.set_xlabel('Annual Volatility (Risk)', fontsize=12)
ax.set_ylabel('Annual Return', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=11)

# Format axes as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

plt.tight_layout()
plt.show()

print("\n📊 Risk-Return Profile")

---

## Cell 9: Drawdown Analysis

In [ ]:
# Calculate cumulative returns
cumulative = (1 + returns).cumprod()

# Calculate running maximum
running_max = cumulative.cummax()

# Calculate drawdown
drawdown = (cumulative - running_max) / running_max

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

for ticker in drawdown.columns:
    ax.fill_between(drawdown.index, 0, drawdown[ticker], alpha=0.3, label=ticker)

ax.set_title('Drawdown Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Drawdown', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

# Format y-axis as percentage
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

plt.tight_layout()
plt.show()

print("\n📉 Maximum Drawdowns:")
for ticker in drawdown.columns:
    max_dd = drawdown[ticker].min()
    print(f"  {ticker}: {max_dd:.2%}")

---

## Cell 10: Summary Report

In [ ]:
print("="*70)
print("📊 BACKTESTING SYSTEM TEST SUMMARY".center(70))
print("="*70)

print("\n✅ System Components Tested:")
print("  1. DataLoader          ✓ Working")
print("  2. RiskMetrics         ✓ Working")
print("  3. PortfolioOptimizer  ✓ Working")
print("  4. Visualization       ✓ Working")

print("\n📊 Data Summary:")
print(f"  ETFs:             {len(etfs)}")
print(f"  Benchmarks:       {len(benchmarks)}")
print(f"  Price History:    {len(prices):,} rows")
print(f"  Date Range:       {prices['price_date'].min()} to {prices['price_date'].max()}")

print("\n📈 Performance Metrics:")
for ticker in returns.columns:
    ann_return = returns[ticker].mean() * 252
    ann_vol = returns[ticker].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    print(f"\n  {ticker}:")
    print(f"    Annual Return:  {ann_return:7.2%}")
    print(f"    Annual Vol:     {ann_vol:7.2%}")
    print(f"    Sharpe Ratio:   {sharpe:7.2f}")

print("\n" + "="*70)
print("🎉 ALL TESTS PASSED! SYSTEM IS READY!".center(70))
print("="*70)

print("\n💡 Next Steps:")
print("  1. Import full dataset (208,700 rows)")
print("  2. Run complete backtesting with main.ipynb")
print("  3. Generate final analysis with analytics.ipynb")
print("  4. Write final report")

print("\n✅ System is working perfectly! Ready for production data.")